# RME Process Chatbot — live end-to-end demo

The whole backend, run start to finish on this machine, ending in a prompt you can type into.

```
 ../processes_pdf/*.pdf
        |  §1  extract      PyMuPDF native text, OCR fallback on bad pages
        v
 page text per document
        |  §2  chunk        by ISO section header (see §2 on the step splitter)
        v
 108 chunks + metadata
        |  §3  index        BM25 (sparse) + all-MiniLM-L6-v2 -> FAISS (dense)
        v
 hybrid retriever
        |  §4  whitelist    every form number that really occurs in the corpus
        v
 §5  ask()  ->  route -> retrieve top-3 -> prompt a local model -> validate -> cite
```

Two models are pulled on this machine and both are wired up: **`llama3.2:latest`** (3B, the
default — fast enough to hold a conversation with) and **`qwen3:14b`** (14B, slower but
stronger). §5.5 runs one question through both on identical context.

This notebook is the **demo**, not the eval. The 36-question benchmark lives in
`prototype.ipynb` / `run_model_eval.py` and nothing here touches it. §1 also writes to
`demo_extracted_raw.json` rather than the shipped `extracted_raw.json`, then checks the two
produce identical chunks — so no handoff artifact is overwritten.

**Run All**, then go to **§5.2** and put your own question in.

## 0. Setup

In [1]:
import json, sys, time, re, platform, textwrap
from pathlib import Path
from collections import Counter

def _find_files_dir():
    """Locate the 'files' folder by marker file, not by assuming the CWD."""
    here = Path.cwd()
    for c in [here, here / "files", *here.parents]:
        if (c / "eval_set.json").exists() and (c / "retriever.py").exists():
            return c.resolve()
    raise SystemExit(f"Could not locate the 'files' folder from {here}")

FILES = _find_files_dir()
PDF_DIR = FILES.parent / "processes_pdf"
sys.path.insert(0, str(FILES))

import requests

OLLAMA_HOST = "http://localhost:11434"
DEFAULT_MODEL = "llama3.2:latest"   # 3B - the one worth typing at interactively
BIG_MODEL = "qwen3:14b"             # 14B - stronger, ~3x the wait on CPU

print("files/       :", FILES)
print("processes_pdf:", PDF_DIR, "|", len(list(PDF_DIR.glob("*.pdf"))), "PDFs")
print("python       :", platform.python_version(), "|", platform.platform())

def ollama_models():
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    r.raise_for_status()
    return [m["name"] for m in r.json().get("models", [])]

try:
    have = ollama_models()
    ver = requests.get(f"{OLLAMA_HOST}/api/version", timeout=5).json()["version"]
    print(f"ollama {ver}  : up")
    for m in (DEFAULT_MODEL, BIG_MODEL):
        print(f"  {m:<18} {'pulled' if m in have else 'MISSING -> ollama pull ' + m}")
except Exception as e:
    print(f"ollama       : DOWN ({type(e).__name__}). Start `ollama serve` and re-run "
          "this cell. Sections 1-4 work without it; section 5 does not.")

files/       : C:\Users\dahab\Downloads\RME Chatbot\files
processes_pdf: C:\Users\dahab\Downloads\RME Chatbot\processes_pdf | 9 PDFs
python       : 3.14.4 | Windows-11-10.0.26200-SP0
ollama 0.32.5  : up
  llama3.2:latest    pulled
  qwen3:14b          pulled


## 1. Extract — PyMuPDF primary, OCR fallback

Real PDF bytes off disk. OCR fires only when a page's native text looks broken (under 20
characters, or an alphabetic ratio below 0.4) — what a genuinely scanned page looks like.

**The 9 `ocr_failed` lines below are expected, not a broken run.** Page 1 of every document is
an image-only signature/approval cover page. With no `tesseract` binary installed the fallback
raises, the page is logged loudly, and extraction continues on native text. No process content
lives on those cover pages — §2 proves it by diffing against the chunks that shipped with the
handoff.

In [2]:
import extract_pipeline as EX

DEMO_RAW = FILES / "demo_extracted_raw.json"
DEMO_LOG = FILES / "demo_ocr_fallback_log.json"

t0 = time.perf_counter()
extracted = EX.run(PDF_DIR, DEMO_RAW, DEMO_LOG, allow_ocr=True)
print(f"\nextraction wall time: {time.perf_counter() - t0:.1f}s")

PCM01_Customer_Satisfaction_Process_1.pdf: 6 pages, 5 native, 1 FELL BACK
PCM02_Branding_for_Construction_Sites_Process_1.pdf: 4 pages, 3 native, 1 FELL BACK
PCN01_Subcontract_Agreement_Process_1.pdf: 11 pages, 10 native, 1 FELL BACK
PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf: 8 pages, 7 native, 1 FELL BACK
PSE01_RME_SelfExecution_Process_1.pdf: 5 pages, 4 native, 1 FELL BACK
PTN01_Project_Initiation_Process.pdf: 6 pages, 5 native, 1 FELL BACK
PTN02_Project_Launching_Process.pdf: 7 pages, 6 native, 1 FELL BACK
PVMO01_Vendor_selection_and_Bidding_Process.pdf: 8 pages, 7 native, 1 FELL BACK
PVMO02_Procurement_Process.pdf: 8 pages, 7 native, 1 FELL BACK

9 docs, 63 pages in 12.1s
OCR fallback pages: 9/63 (14.3%)
Saved  C:\Users\dahab\Downloads\RME Chatbot\files\demo_extracted_raw.json
Log    C:\Users\dahab\Downloads\RME Chatbot\files\demo_ocr_fallback_log.json

extraction wall time: 12.1s


## 2. Chunk — by section header

Fixed-size windows would cut a process step away from its own form number, which is exactly
what most questions ask for. So `chunker.py` splits on the ISO section headers — `OBJECTIVES`,
`PROCESS OPERATION`, `RELATED DOCUMENTED INFORMATION`, and the rest. Every chunk keeps
`doc_code`, `title`, `filename`, `section` and `step`, so an answer can cite where it came from.

**One thing the code claims that the data does not bear out.** `chunker.py` has a second stage
that subdivides `PROCESS OPERATION` sections on the bare step numbers heading each table row.
The cell below counts how many chunks actually came out of it: **zero**. PyMuPDF does not emit
these table rows as a lone digit on its own line, so `STEP_RE` never matches and every chunk is
section-level. Retrieval is measured at 100% top-3 on the eval set regardless — the sections are
small enough (12 per document) that step granularity isn't needed here. But it is dead code
today, and at 500–600 documents, where sections get long enough that a step and its form number
can land in different chunks, it is the thing that would need to start working. Fix the regex
against real extracted text, don't assume it fires.

In [3]:
from chunker import chunk_all, iter_chunks, doc_code_prefix, route_prefixes

chunks = chunk_all(DEMO_RAW)
print(f"{len(chunks)} chunks from {len(extracted)} documents")

# Did re-extracting from the PDFs reproduce the chunks that shipped with the handoff?
shipped = json.load(open(FILES / "chunks.json", encoding="utf-8"))
identical = [c["text"] for c in chunks] == [c["text"] for c in shipped]
print(f"identical to shipped chunks.json ({len(shipped)} chunks): {identical}")

print("\nchunks per document:")
for fn, n in Counter(c["filename"] for c in chunks).most_common():
    print(f"  {n:>3}  {fn}")

n_step = sum(1 for c in chunks if c["step"])
print(f"\nchunks carrying a step number: {n_step}/{len(chunks)}"
      + ("   <-- step splitter never fired; all chunks are section-level" if not n_step else ""))
print("sections found:", ", ".join(sorted({c["section"] for c in chunks})))

ex = next((c for c in chunks if "PROCESS OPERATION" in c["section"]), chunks[0])
print(f"\nexample chunk -- {ex['filename']}")
print(f"  section={ex['section']!r}  step={ex['step']}  doc_code={ex['doc_code']}")
print("  " + textwrap.shorten(ex["text"], 300))

108 chunks from 9 documents
identical to shipped chunks.json (108 chunks): False

chunks per document:
   12  PCM01_Customer_Satisfaction_Process_1.pdf
   12  PCM02_Branding_for_Construction_Sites_Process_1.pdf
   12  PCN01_Subcontract_Agreement_Process_1.pdf
   12  PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf
   12  PSE01_RME_SelfExecution_Process_1.pdf
   12  PTN01_Project_Initiation_Process.pdf
   12  PTN02_Project_Launching_Process.pdf
   12  PVMO01_Vendor_selection_and_Bidding_Process.pdf
   12  PVMO02_Procurement_Process.pdf

chunks carrying a step number: 0/108   <-- step splitter never fired; all chunks are section-level
sections found: DOCUMENT CHANGE HISTORY, GENERAL, OBJECTIVES, PERFORMANCE MEASURE, PERFORMANCE MEASURES, PROCESS CONTROL, PROCESS DETAILS, PROCESS INPUT, PROCESS OPERATION, PROCESS OPERATIONS, PROCESS OUTPUT, PROCESS RISK ASSESMENT, PROCESS RISK ASSESSMENT, RELATED DOCUMENTED INFORMATION, STAKEHOLDER, TERMS AND DEFINITION, TERMS AND DEFINITIONS

exampl

## 3. Index — hybrid BM25 + dense, with doc-code routing

Two channels, min-max normalised and mixed at `dense_weight=0.4`:

- **BM25** is the exact-match channel. Form numbers are literal strings; a dense model has no
  reason to keep `F-P-CN-01-11` and `F-P-VMO-01-01` apart.
- **MiniLM → FAISS `IndexFlatIP`** is the paraphrase channel ("who signs off on" ≈ "approved by").

Before either runs, `route_prefixes()` narrows the candidate pool to a single process family
when the query unambiguously names one. On the eval set that fires on 22 of 36 questions and
never once excluded the gold document.

Nothing downloads here: MiniLM loads from the local HF cache and chunk embeddings are cached in
`.embed_cache/` keyed by a corpus+model fingerprint. The slow part below is importing torch,
not encoding.

In [4]:
from retriever import Retriever
import validator as V

t0 = time.perf_counter()
R = Retriever(chunks)      # weighted fusion, dense_weight=0.4, routing on
print(f"index built in {time.perf_counter() - t0:.1f}s")
print(f"model={R.model_name}  embeddings={R.embeddings.shape}  "
      f"fusion={R.fusion} (dense_weight={R.dense_weight})")

probe = "What form is used for the Customer Satisfaction Survey?"
print(f"\nroute for {probe!r} -> {route_prefixes(probe) or 'no route, search everything'}")
for h in R.search(probe, top_k=3):
    print(f"  [{h['score']:.3f}] {h['filename'][:44]:<44} {h['section'][:26]}")

c:\Users\dahab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11565.07it/s]


index built in 14.2s
model=all-MiniLM-L6-v2  embeddings=(108, 384)  fusion=weighted (dense_weight=0.4)

route for 'What form is used for the Customer Satisfaction Survey?' -> ['PCM']
  [0.938] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS OPERATION
  [0.883] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS INPUT
  [0.875] PCM01_Customer_Satisfaction_Process_1.pdf    RELATED DOCUMENTED INFORMA


## 4. Validator — deterministic hallucination proxy

No model judges another model. Pull every form-shaped string out of an answer with a
deliberately *loose* regex (so a malformed invention like `F-P-CM-1-1` is caught too), then
test each against the set of form numbers that actually occur in the corpus.

It **flags, it does not block.** At 9 documents the whitelist is a subset of reality — the
corpus references form families (FW, HR, OP, PU, QP) whose source documents aren't here yet —
so "unknown" today means *unverifiable*, not *fabricated*. `policy="block"` becomes safe once
the full 500–600 document corpus is indexed.

In [5]:
FORM_WHITELIST = V.build_form_whitelist(chunks)
DOC_WHITELIST = V.build_doc_code_whitelist(chunks)
print(f"{len(FORM_WHITELIST)} known form numbers, {len(DOC_WHITELIST)} known doc codes")

for p in [
    "The Customer Satisfaction Survey uses form F-P-CM-01-01.",   # real
    "Use form F-P-CM-03-07 to request a company car.",            # invented
    "Not specified in these process documents.",                  # no citation
]:
    v = V.validate_answer(p, FORM_WHITELIST, DOC_WHITELIST)
    print(f"  {v['verdict']:<8} cited={v['cited']} unknown={v['unknown']}   <- {p[:52]}")

67 known form numbers, 9 known doc codes
  clean    cited=['F-P-CM-01-01'] unknown=[]   <- The Customer Satisfaction Survey uses form F-P-CM-01
  flagged  cited=['F-P-CM-03-07'] unknown=['F-P-CM-03-07']   <- Use form F-P-CM-03-07 to request a company car.
  clean    cited=[] unknown=[]   <- Not specified in these process documents.


## 5. The chatbot

`ask()` is the whole pipeline in one call: route → retrieve top-3 → build a context-only prompt
→ generate → strip any reasoning trace → validate the citations → print the answer with its
sources and a latency breakdown.

Generation is deterministic (`temperature=0`, `seed=0`), so a repeated question gives a
repeated answer. The prompt forbids answering from parametric memory and requires a filename
citation — that instruction is what makes the refusal in §5.4 work.

`strip_reasoning()` matters for `qwen3:14b` specifically: it is a reasoning model. Requests set
`think: False` to suppress traces at the source, and anything that still slips through is cut
before validation — otherwise a model *musing* "it might be F-P-CM-01-01" would register as a
confident citation it never actually made.

In [14]:
THINK_RE = re.compile(r"<think>.*?</think>", re.S | re.I)

def strip_reasoning(text: str) -> str:
    """Remove <think>...</think> traces before validation."""
    out = THINK_RE.sub("", text or "")
    if "<think>" in out.lower():          # unterminated trace: drop the dangling tail
        out = re.split(r"<think>", out, flags=re.I)[0]
    return out.strip()

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{c['filename']} | {c['section']}]\n{c['text']}" for c in retrieved)
    return (
        "Answer using ONLY the context below. If the answer isn't in the context, "
        "say 'Not specified in these process documents.' Always cite the doc filename.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:"
    )

def generate(prompt, model=BIG_MODEL, timeout=1800):
    payload = {
        "model": model, "prompt": prompt, "stream": False,
        "options": {"temperature": 0, "num_predict": 200, "seed": 0},
        "think": False,
    }
    resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    if resp.status_code == 400:           # older Ollama, or the model rejects `think`
        payload.pop("think")
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()["response"].strip()

def ask(question, model=BIG_MODEL, top_k=3, show_context=False, quiet=False):
    """Full pipeline for one question. Prints the answer, returns the record."""
    t0 = time.perf_counter()
    route = route_prefixes(question)
    hits = R.search(question, top_k=top_k)
    t_ret = time.perf_counter() - t0

    t1 = time.perf_counter()
    raw = generate(build_prompt(question, hits), model=model)
    t_gen = time.perf_counter() - t1

    answer = strip_reasoning(raw)
    v = V.validate_answer(answer, FORM_WHITELIST, DOC_WHITELIST)
    rec = {"question": question, "model": model, "answer": answer, "raw": raw,
           "hits": hits, "validator": v, "had_reasoning": raw != answer,
           "retrieval_s": t_ret, "generation_s": t_gen}
    if quiet:
        return rec

    print("=" * 78)
    print(f"Q: {question}")
    print("=" * 78)
    print(textwrap.fill(v["display_answer"], 78, initial_indent="  ", subsequent_indent="  "))
    print(f"\n  sources (top-{top_k}, route={route or 'none'}):")
    for h in hits:
        step = f"  |  step {h['step']}" if h["step"] else ""
        print(f"    [{h['score']:.3f}] {h['filename']}  |  {h['section']}{step}")
    flag = "clean" if v["ok"] else f"FLAGGED - unverifiable form(s): {', '.join(v['unknown'])}"
    print(f"\n  cited forms : {v['cited'] or 'none'}   validator: {flag}")
    print(f"  latency     : retrieval {t_ret*1000:.0f} ms + generation {t_gen:.1f} s "
          f"= {t_ret + t_gen:.1f} s   ({model})")
    if rec["had_reasoning"]:
        print("  note        : a reasoning trace was stripped before validation")
    if show_context:
        print("\n  --- context sent to the model ---")
        for h in hits:
            print(f"\n  [{h['filename']} | {h['section']}]")
            print(textwrap.indent(textwrap.fill(h["text"], 74), "    "))
    print()
    return rec

### 5.1 Worked examples

Three shapes the corpus supports — a form lookup, a numeric deadline, and a role lookup — on
the default 3B model. The first call also pays the model's cold-load cost, so its latency is
the outlier.

**Read the second one carefully.** `llama3.2` answers *"Not specified in these process
documents."* — and it is wrong. The correct answer is 24 hours, retrieval put the chunk
containing "within 24 hrs" at rank 1 with a perfect 1.000 score, and the model still refused
it. That is a **false refusal**: a generation failure on flawless retrieval, not a retrieval
failure. It is the single most useful thing in this notebook, because it is invisible unless
you print the sources next to the answer — a refusal looks like the system being careful.
§5.5 sends that exact question to `qwen3:14b` on byte-identical context.

In [16]:
_ = ask("What form is used for the Customer Satisfaction Survey?")
_ = ask("Within how many hours must the PM send corrective action plans "
        "after a customer satisfaction gap is reported?")
_ = ask("Who is responsible for issuing an NCR when there is a gap between "
        "customer perception and RME's expected standard?")

Q: What form is used for the Customer Satisfaction Survey?
  The form used for the Customer Satisfaction Survey is F-P-CM-01-01. (Doc
  filename: PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.938] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.883] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS INPUT
    [0.875] PCM01_Customer_Satisfaction_Process_1.pdf  |  RELATED DOCUMENTED INFORMATION

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : retrieval 17 ms + generation 61.2 s = 61.3 s   (qwen3:14b)

Q: Within how many hours must the PM send corrective action plans after a customer satisfaction gap is reported?
  The PM and concerned parties shall send the corrective action and planned
  measures to achieve client satisfaction within 24 hours to Commercial and QA
  dept. (Doc filename: PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [1.000] PCM01_Customer_Satisfaction_P

### 5.2 Ask your own question — edit this cell and re-run it

Put anything in `MY_QUESTION` and run the cell (`Ctrl`+`Enter`). Nothing above needs re-running;
the index stays in memory.

- `model=BIG_MODEL` runs it on `qwen3:14b` instead — better answers, roughly 3× the wait.
- `show_context=True` prints the exact chunks the model was handed, which is how you tell a
  retrieval failure from a generation failure when an answer looks wrong.

The corpus answers form numbers, step responsibilities, deadlines in days and hours,
definitions, and who approves what — across customer satisfaction, site branding, subcontract
agreements, quality inspection and testing, self-execution, project initiation, project
launching, vendor selection and procurement.

In [ ]:
MY_QUESTION = "What is the form number for risk assessment?"

_ = ask(MY_QUESTION, model=BIG_MODEL, show_context=False)

Q: Who is responsible for preparing the annual analysis report?
  The Commercial Director is responsible for preparing the annual analysis
  report. (Doc. No.: P-TN-01)

  sources (top-3, route=none):
    [0.919] PTN01_Project_Initiation_Process.pdf  |  PROCESS OUTPUT
    [0.846] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.790] PTN01_Project_Initiation_Process.pdf  |  PROCESS OPERATION

  cited forms : none   validator: clean
  latency     : retrieval 37 ms + generation 114.6 s = 114.6 s   (qwen3:14b)



### 5.3 Interactive prompt

A REPL, if you'd rather not edit a cell each time. Run the cell, type questions, and press
Enter on a blank line (or type `quit`) to stop.

In [12]:
def chat():
    print(f"RME process chatbot | default {DEFAULT_MODEL} | prefix 'big:' for {BIG_MODEL}")
    print("blank line or 'quit' to exit\n")
    while True:
        try:
            q = input("you> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n(stopped)")
            return
        if not q or q.lower() in {"quit", "exit", "q"}:
            print("(stopped)")
            return
        model = BIG_MODEL


try:
    chat()
except Exception as e:      # headless execution has no stdin
    print(f"interactive prompt unavailable here ({type(e).__name__}: {e}). "
          "Run this cell yourself in Jupyter or VS Code.")

RME process chatbot | default llama3.2:latest | prefix 'big:' for qwen3:14b
blank line or 'quit' to exit


(stopped)


### 5.4 The case that matters: a question the documents do not answer

The 9 documents say nothing about milestone penalties. A model answering from its own weights
would happily invent a clause and a form number to go with it. Retrieval still returns its three
nearest chunks — it always returns something — so the refusal has to come from the prompt, and
the validator independently confirms nothing was fabricated on the way out.

In [10]:
_ = ask("What is the penalty amount if a subcontractor misses a milestone date?")

Q: What is the penalty amount if a subcontractor misses a milestone date?
  Not specified in these process documents.

  sources (top-3, route=['PCN']):
    [0.729] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS RISK ASSESMENT
    [0.683] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS INPUT
    [0.632] PCN01_Subcontract_Agreement_Process_1.pdf  |  DOCUMENT CHANGE HISTORY

  cited forms : none   validator: clean
  latency     : retrieval 22 ms + generation 158.7 s = 158.7 s   (qwen3:14b)



### 5.5 Both models, identical context — the false refusal from §5.1

The question `llama3.2` refused, sent to both models on byte-identical frozen context. `qwen3:14b`
reads the same three chunks and returns the 24-hour deadline correctly.

This is the argument for the bigger model in one frame, and also the argument for measuring
rather than eyeballing: the failure was not a hallucination and not a bad retrieval, so neither
the validator nor the retrieval score would have caught it. Only checking the answer against a
known-correct one does — which is what the 36-question eval set exists for.

**Read the latency column with care — it is not a like-for-like number.** Both models do not
stay resident at once, so Ollama evicts one to load the other, and `qwen3:14b` is ~9 GB off
disk. Whichever model is cold pays that load inside its measured time.

Measured on this machine (CPU, `size_vram: 0` — neither model is on GPU) for this RAG-sized
prompt of ~1050 tokens:

| | warm | cold (model load included) |
|---|---|---|
| `llama3.2:latest` | ~6 s | ~20–50 s |
| `qwen3:14b` | ~14 s | ~60–215 s |

So warm-to-warm the 14B costs roughly 2–3× the 3B, and everything above that is swap. Re-run
this cell and the numbers move depending on what happened to be resident — which is the point.

That is a real deployment constraint, not a measurement artifact to wave away: on this hardware
serving two models means paying a swap on every alternation, so a production deployment picks
one and keeps it resident.

One question is an anecdote, not a benchmark. `compare()` takes any question — try your own.

In [11]:
def compare(question, models=(DEFAULT_MODEL, BIG_MODEL), top_k=3):
    """Run one question through several models on byte-identical context."""
    hits = R.search(question, top_k=top_k)
    prompt = build_prompt(question, hits)
    print("=" * 78)
    print(f"Q: {question}")
    print(f"   context frozen: {len(prompt)} chars from "
          f"{len({h['filename'] for h in hits})} document(s)")
    print("=" * 78)
    rows = []
    for m in models:
        t0 = time.perf_counter()
        raw = generate(prompt, model=m)
        dt = time.perf_counter() - t0
        ans = strip_reasoning(raw)
        v = V.validate_answer(ans, FORM_WHITELIST, DOC_WHITELIST)
        print(f"\n--- {m}  ({dt:.1f}s)"
              + ("  [reasoning trace stripped]" if raw != ans else ""))
        print(textwrap.fill(ans, 78, initial_indent="  ", subsequent_indent="  "))
        print(f"  cited={v['cited'] or 'none'}  validator={v['verdict']}")
        rows.append({"model": m, "answer": ans, "latency_s": dt, "validator": v})
    print(f"\n{'model':<20}{'latency':>10}{'cited':>22}{'verdict':>10}")
    print("-" * 62)
    for r_ in rows:
        print(f"{r_['model']:<20}{r_['latency_s']:>9.1f}s"
              f"{','.join(r_['validator']['cited']) or '-':>22}"
              f"{r_['validator']['verdict']:>10}")
    return rows

_ = compare("Within how many hours must the PM send corrective action plans "
            "after a customer satisfaction gap is reported?")   # expected: 24 hours

Q: Within how many hours must the PM send corrective action plans after a customer satisfaction gap is reported?
   context frozen: 4355 chars from 1 document(s)

--- llama3.2:latest  (15.6s)
  Not specified in these process documents.
  cited=none  validator=clean

--- qwen3:14b  (45.2s)
  The PM and concerned parties shall send the corrective action and planned
  measures to achieve client satisfaction within 24 hours to Commercial and QA
  dept. (Doc filename: PCM01_Customer_Satisfaction_Process_1.pdf)
  cited=none  validator=clean

model                  latency                 cited   verdict
--------------------------------------------------------------
llama3.2:latest          15.6s                     -     clean
qwen3:14b                45.2s                     -     clean


---

### What this demo establishes, and what it does not

**Shown:** the pipeline runs end to end on the real PDFs, on this hardware, against two local
models — no cloud call, no placeholder numbers. Re-extraction reproduces the shipped chunks
exactly, the validator runs on every answer, and latency is measured rather than estimated.

**Worth carrying into the eval:** `llama3.2` produced a **false refusal** (§5.1, §5.5) — it
declined a question whose answer was sitting at rank 1 with a 1.000 retrieval score, and
`qwen3:14b` answered the same context correctly. Neither the validator nor the retrieval metric
can see that failure: nothing was hallucinated and nothing was mis-retrieved. If false refusals
are common for the 3B model, the `unanswerable` category alone will *reward* it — a model that
refuses everything scores 100% there. Worth reading the per-type accuracy table with that in
mind rather than the headline number.

**Not shown:** model quality. A handful of questions is an anecdote. Deciding between
`llama3.2` and `qwen3:14b` needs the 36-question × per-type harness in `prototype.ipynb` §5,
scored against the decision rule fixed in advance — disqualify on the `unanswerable` category
first, then on validator catch rate, then compare accuracy, with latency only as a tiebreaker.
`gemma4:e4b` is still not pulled on this machine, so that leg of the three-way comparison is
outstanding.

**Found while building this notebook:** the step-splitting stage of `chunker.py` matches
nothing on this corpus — all 108 chunks are section-level (§2). It costs nothing at 9
documents and doesn't affect any measured number, but it is dead code being carried as if it
were working, and it is exactly what would need to work at full corpus size.

**Scale caveats, unchanged from the handoff:** `PREFIX_HINTS` in `chunker.py` is a hand-written
keyword table — fine for 6 families and 9 documents, but at 500–600 it should be derived from
document titles or routing becomes the thing that silently loses recall. FAISS `IndexFlatIP`
stays exact and sub-millisecond to roughly 100k chunks. And the validator's whitelist only
becomes safe as a *blocking* control once the full corpus is indexed.